# CIN and CLA Pathways

Tracing the statutory children's social care pathway in England — referral → assessment → Child in Need (CIN) → Child Protection Plan (CPP) → Children Looked After (CLA) — using the Department for Education's published child-level census data (Explore Education Statistics).

This is a companion piece to an earlier SEN pathways notebook, using the same approach: pull the DfE's own local-authority-level statistics, join them into a single pathway, and see what the funnel actually looks like.

**A note on scope:** this notebook stops at CLA. Emergency Protection Orders are a family court (Cafcass) matter, not part of the Children in Need census, so they sit just outside what this data can show — more on that in the final section.

## Sections
1. Load and inspect the source data
2. Clean and align local authority boundaries across years
3. Build the pathway (referral → assessed → CIN → CPP → CLA)
4. Explore variation by local authority and over time
5. Where EPOs sit outside this data

## 1. Load and inspect the source data

Starting with C1 — referrals and re-referrals to children's social care services by local authority. No transformation yet, just loading it and looking at what we've actually got.

In [ ]:
import pandas as pd

# C1: Referrals and re-referrals to children's social care services by local authority
c1_url = "https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/306c2a56-228b-43f4-8493-894bee427a8c/csv"

df_referrals = pd.read_csv(c1_url)

print(df_referrals.shape)
df_referrals.head()

In [ ]:
df_referrals.columns.tolist()

## 2. Clean and align local authority boundaries across years

Both datasets span local government reorganisation (LGR) years, so the same local authority can appear under different codes depending on the year and which DfE publication you're looking at. Two distinct problems, needing two distinct fixes:

**Continuing authorities** (Buckinghamshire, North Yorkshire, Somerset) — the new unitary council is legally the same body as the old county council, just renamed when the surrounding district councils were abolished into it. Since children's social care was always a county-level function, there's no real merger of caseloads here — just a code relabelling that happens on a different timetable in different DfE files. Fix: join on `old_la_code`, which stays stable across the rename, instead of `new_la_code`, which doesn't.

**Genuine splits** (Northamptonshire → North/West Northamptonshire; Cumbria → Cumberland/Westmorland and Furness) — these really did divide one caseload into two new authorities that never separately reported it before. There's no code to anchor on. Fix: apportion the combined-authority figures by population share, calculated from the earliest year both successor authorities report separately.

In [ ]:
def apportion_row(row, share_a, share_b, name_a, name_b):
    """Split a combined-authority row into two new rows using a fixed population share."""
    original_number = int(row["number"])

    row_a = row.copy()
    row_a["la_name"] = name_a
    row_a["number"] = round(original_number * share_a)

    row_b = row.copy()
    row_b["la_name"] = name_b
    row_b["number"] = round(original_number * share_b)

    return row_a, row_b

### Northamptonshire split

Ratio anchored on 2022 — the first year North and West Northamptonshire report population separately (2020/2021 population is DfE-suppressed).

In [ ]:
north_pop, west_pop = 79186, 92050
north_share = north_pop / (north_pop + west_pop)
west_share = west_pop / (north_pop + west_pop)

print(f"North Northamptonshire share: {north_share:.4f}")
print(f"West Northamptonshire share: {west_share:.4f}")

### Cumbria split

Ratio anchored on 2024 — the first year Cumberland and Westmorland and Furness report population separately (2020–2023 population is DfE-suppressed).

In [ ]:
cumberland_pop, westmorland_pop = 51775, 38971
cumberland_share = cumberland_pop / (cumberland_pop + westmorland_pop)
westmorland_share = westmorland_pop / (cumberland_pop + westmorland_pop)

print(f"Cumberland share: {cumberland_share:.4f}")
print(f"Westmorland and Furness share: {westmorland_share:.4f}")

### Apply both splits to the CLA snapshot

Years needing apportionment (identified by checking which years show real numbers for the combined county alongside suppressed (`z`/`x`) figures for the successor authorities):
- Northamptonshire: 2020, 2021
- Cumbria: 2020, 2021, 2022, 2023

2022–2024 for Northamptonshire and 2024 for Cumbria already report the split authorities separately in this dataset — no apportionment needed there, DfE has done it for us.

In [ ]:
def apportion_years(df, county_name, years, share_a, share_b, name_a, name_b):
    combined = df[(df["la_name"] == county_name) & (df["time_period"].isin(years))]
    new_rows = []
    for _, row in combined.iterrows():
        a, b = apportion_row(row, share_a, share_b, name_a, name_b)
        new_rows.extend([a, b])
    return pd.DataFrame(new_rows)

northants_apportioned = apportion_years(
    cla_snapshot, "Northamptonshire", [2020, 2021],
    north_share, west_share, "North Northamptonshire", "West Northamptonshire"
)

cumbria_apportioned = apportion_years(
    cla_snapshot, "Cumbria", [2020, 2021, 2022, 2023],
    cumberland_share, westmorland_share, "Cumberland", "Westmorland and Furness"
)

print(northants_apportioned[["time_period", "la_name", "number"]])
print(cumbria_apportioned[["time_period", "la_name", "number"]])

### Rebuild the CLA snapshot with apportioned rows included

Drop the now-redundant combined-county rows for the apportioned years, append the new split rows in their place.

In [ ]:
cla_snapshot_fixed = cla_snapshot[
    ~(
        ((cla_snapshot["la_name"] == "Northamptonshire") & (cla_snapshot["time_period"].isin([2020, 2021]))) |
        ((cla_snapshot["la_name"] == "Cumbria") & (cla_snapshot["time_period"].isin([2020, 2021, 2022, 2023])))
    )
]

cla_snapshot_fixed = pd.concat(
    [cla_snapshot_fixed, northants_apportioned, cumbria_apportioned],
    ignore_index=True
)

print(cla_snapshot.shape[0], "-> ", cla_snapshot_fixed.shape[0], "rows after apportionment")

### Rebuild the pathway join with the fixed data

Using `old_la_code` (stable across continuing-authority renames) plus the newly apportioned split rows. This is the join to check against the original 750/759 figures.

In [ ]:
pathway_v3 = la_rows.merge(
    cla_snapshot_fixed,
    on=["old_la_code", "time_period"],
    how="inner",
    suffixes=("_referrals", "_cla")
)

print(la_rows.shape[0], "referral rows")
print(cla_snapshot_fixed.shape[0], "CLA rows (post-apportionment)")
print(pathway_v3.shape[0], "joined rows")

**Note on scope:** the referrals dataset (C1) reports "Northamptonshire" as a combined figure all the way from 2013–2021 (versus CLA's shorter 2020–2021 combined window), and Cumbria similarly. The same `old_la_code` join won't recover those years on its own — the referrals side needs the same apportionment treatment applied separately, using the same two population ratios calculated above, before the pathway is fully resolved back to 2013. That's the next step.

**Methodological caveat, kept visible rather than buried in a footnote:** apportioning by population share assumes referral and CLA counts scale with headcount. In reality, deprivation — not population alone — drives referral rates, and the two halves of a split county aren't necessarily equally deprived. This is a reasonable, documented simplification for a retrospective estimate, not a claim of precision.

### Apportion the referrals side

Same population ratios as the CLA split, but a longer combined-year run in this dataset than in CLA — Northamptonshire reports combined 2013–2021 here (vs 2020–2021 in CLA), and Cumbria 2013–2023 (vs 2020–2023 in CLA). Confirmed by checking each county's combined-vs-split years directly rather than assuming the two datasets share a cutover point — they don't.

`apportion_row` was hardcoded to the CLA `number` column, so it's generalised here into `apportion_years_col`, which takes the target column as an argument. Both functions do the same job; keeping `apportion_row` as-is above rather than retrofitting it, since the CLA pipeline already runs against it and there's no benefit to touching working code mid-flow.

In [ ]:
def apportion_years_col(df, county_name, years, share_a, share_b, name_a, name_b, col):
    combined = df[(df["la_name"] == county_name) & (df["time_period"].isin(years))]
    new_rows = []
    for _, row in combined.iterrows():
        original = int(row[col])
        row_a = row.copy()
        row_a["la_name"] = name_a
        row_a[col] = round(original * share_a)

        row_b = row.copy()
        row_b["la_name"] = name_b
        row_b[col] = round(original * share_b)

        new_rows.extend([row_a, row_b])
    return pd.DataFrame(new_rows)

northants_ref_apportioned = apportion_years_col(
    la_rows, "Northamptonshire", list(range(2013, 2022)),
    north_share, west_share, "North Northamptonshire", "West Northamptonshire", "Referrals"
)

cumbria_ref_apportioned = apportion_years_col(
    la_rows, "Cumbria", list(range(2013, 2024)),
    cumberland_share, westmorland_share, "Cumberland", "Westmorland and Furness", "Referrals"
)

print(northants_ref_apportioned[["time_period", "la_name", "Referrals"]].shape[0], "Northants rows")
print(cumbria_ref_apportioned[["time_period", "la_name", "Referrals"]].shape[0], "Cumbria rows")

### Rebuild the referrals table with apportioned rows included

Same pattern as the CLA rebuild: drop the now-redundant combined-county rows for the apportioned years, append the split rows in their place.

In [ ]:
la_rows_fixed = la_rows[
    ~(
        ((la_rows["la_name"] == "Northamptonshire") & (la_rows["time_period"].isin(range(2013, 2022)))) |
        ((la_rows["la_name"] == "Cumbria") & (la_rows["time_period"].isin(range(2013, 2024))))
    )
]

la_rows_fixed = pd.concat(
    [la_rows_fixed, northants_ref_apportioned, cumbria_ref_apportioned],
    ignore_index=True
)

print(la_rows.shape[0], "-> ", la_rows_fixed.shape[0], "rows after apportionment")

### Final rebuilt pathway join

Both datasets now have Northamptonshire and Cumbria fully resolved into their current successor authorities across the whole 2013–2024 run, plus the `old_la_code` fix for the three continuing-authority cases (Buckinghamshire, North Yorkshire, Somerset). This is the join to check against every previous count: 750 (original) → 759 (old_la_code fix) → this.

In [ ]:
pathway_final = la_rows_fixed.merge(
    cla_snapshot_fixed,
    on=["old_la_code", "time_period"],
    how="inner",
    suffixes=("_referrals", "_cla")
)

print(la_rows_fixed.shape[0], "referral rows")
print(cla_snapshot_fixed.shape[0], "CLA rows")
print(pathway_final.shape[0], "joined rows")

**Expected outcome:** since `old_la_code` is genuinely unique per successor authority for Northamptonshire and Cumbria (unlike the continuing-authority cases, where old and new share history), the apportioned rows should join cleanly against each other now that both sides use the same split names for the same years. If the count comes out at 775 — the full CLA snapshot size — every row has found a match. Worth checking rather than assuming, same as every step so far.

## 3. Bug found and fixed: apportioned rows kept the parent's `old_la_code`

Both apportion functions copied `row` and only overwrote `la_name` and the count column — `old_la_code` was left as the old parent county's code (928 for both Northants successors, 909 for both Cumbria successors) instead of each successor's own real code.

Harmless when only one side of a join was apportioned. Broken in years where **both** referrals and CLA were apportioned for the same county (Northants 2020–2021, Cumbria 2020–2023): both successor rows shared one join key on each side, so the merge produced all four combinations instead of two — North's referrals paired with West's CLA numbers, and vice versa. This is what caused 777 instead of the expected 781.

Fix: assign each successor's real `old_la_code` explicitly, confirmed from the raw data earlier — North Northants 940, West Northants 941, Cumberland 942, Westmorland and Furness 943.

In [ ]:
SUCCESSOR_CODES = {
    "North Northamptonshire": 940,
    "West Northamptonshire": 941,
    "Cumberland": 942,
    "Westmorland and Furness": 943,
}

def apportion_row(row, share_a, share_b, name_a, name_b):
    """Split a combined-authority row into two new rows using a fixed population share."""
    original_number = int(row["number"])

    row_a = row.copy()
    row_a["la_name"] = name_a
    row_a["old_la_code"] = SUCCESSOR_CODES[name_a]
    row_a["number"] = round(original_number * share_a)

    row_b = row.copy()
    row_b["la_name"] = name_b
    row_b["old_la_code"] = SUCCESSOR_CODES[name_b]
    row_b["number"] = round(original_number * share_b)

    return row_a, row_b


def apportion_years_col(df, county_name, years, share_a, share_b, name_a, name_b, col):
    combined = df[(df["la_name"] == county_name) & (df["time_period"].isin(years))]
    new_rows = []
    for _, row in combined.iterrows():
        original = int(row[col])
        row_a = row.copy()
        row_a["la_name"] = name_a
        row_a["old_la_code"] = SUCCESSOR_CODES[name_a]
        row_a[col] = round(original * share_a)

        row_b = row.copy()
        row_b["la_name"] = name_b
        row_b["old_la_code"] = SUCCESSOR_CODES[name_b]
        row_b[col] = round(original * share_b)

        new_rows.extend([row_a, row_b])
    return pd.DataFrame(new_rows)

### Re-run the full pipeline with the fixed functions

Everything downstream of the two functions needs re-running in order: both CLA splits, both referral splits, both table rebuilds, then the final join.

In [ ]:
# CLA side
northants_apportioned = apportion_years(
    cla_snapshot, "Northamptonshire", [2020, 2021],
    north_share, west_share, "North Northamptonshire", "West Northamptonshire"
)
cumbria_apportioned = apportion_years(
    cla_snapshot, "Cumbria", [2020, 2021, 2022, 2023],
    cumberland_share, westmorland_share, "Cumberland", "Westmorland and Furness"
)

cla_snapshot_fixed = cla_snapshot[
    ~(
        ((cla_snapshot["la_name"] == "Northamptonshire") & (cla_snapshot["time_period"].isin([2020, 2021]))) |
        ((cla_snapshot["la_name"] == "Cumbria") & (cla_snapshot["time_period"].isin([2020, 2021, 2022, 2023])))
    )
]
cla_snapshot_fixed = pd.concat([cla_snapshot_fixed, northants_apportioned, cumbria_apportioned], ignore_index=True)

# Referrals side
northants_ref_apportioned = apportion_years_col(
    la_rows, "Northamptonshire", list(range(2013, 2022)),
    north_share, west_share, "North Northamptonshire", "West Northamptonshire", "Referrals"
)
cumbria_ref_apportioned = apportion_years_col(
    la_rows, "Cumbria", list(range(2013, 2024)),
    cumberland_share, westmorland_share, "Cumberland", "Westmorland and Furness", "Referrals"
)

la_rows_fixed = la_rows[
    ~(
        ((la_rows["la_name"] == "Northamptonshire") & (la_rows["time_period"].isin(range(2013, 2022)))) |
        ((la_rows["la_name"] == "Cumbria") & (la_rows["time_period"].isin(range(2013, 2024))))
    )
]
la_rows_fixed = pd.concat([la_rows_fixed, northants_ref_apportioned, cumbria_ref_apportioned], ignore_index=True)

# Final join
pathway_final = la_rows_fixed.merge(
    cla_snapshot_fixed,
    on=["old_la_code", "time_period"],
    how="inner",
    suffixes=("_referrals", "_cla")
)

print(la_rows_fixed.shape[0], "referral rows")
print(cla_snapshot_fixed.shape[0], "CLA rows")
print(pathway_final.shape[0], "joined rows")

Expected: 781, matching the full CLA snapshot row count — every row resolved to a clean 1:1 join. Worth re-checking the 2020/Northants case directly (same query as before) to confirm the fix actually removed the cross-matching rather than just changing the total by coincidence.

### Remaining gap explained: leftover suppressed rows under old county names

777 joined rows, not 781. The gap isn't a bug — it's four leftover rows in the CLA data: a `z`-suppressed row still sitting under "Northamptonshire" (2022, 2023, 2024) and "Cumbria" (2024), left over from years where the real figures have already moved to the split successor authorities. These were correctly excluded from apportionment earlier (the successor authorities already carry real numbers in those years), but never explicitly dropped — so they sat in the table as dead weight that could never find a referrals match, since referrals never had a combined-county row for those years either.

Dropped explicitly here rather than left as an unexplained shortfall in the join count.

In [ ]:
leftover_suppressed = cla_snapshot_fixed[
    (cla_snapshot_fixed["la_name"].isin(["Northamptonshire", "Cumbria"])) &
    (cla_snapshot_fixed["number"] == "z")
]
print(leftover_suppressed.shape[0], "leftover suppressed rows to drop")

cla_snapshot_fixed = cla_snapshot_fixed[~cla_snapshot_fixed.index.isin(leftover_suppressed.index)]
print(cla_snapshot_fixed.shape[0], "CLA rows remaining")

pathway_final = la_rows_fixed.merge(
    cla_snapshot_fixed,
    on=["old_la_code", "time_period"],
    how="inner",
    suffixes=("_referrals", "_cla")
)
print(pathway_final.shape[0], "final joined rows")

## Crosswalk complete: 777 clean rows, 2013–2024, every gap explained

Summary of the full boundary-alignment pipeline:

| Stage | Rows |
|---|---|
| Original naive join (on `new_la_code`) | 750 |
| Fixed to join on `old_la_code` (recovers Bucks/N.Yorks/Somerset renames) | 759 |
| Both datasets apportioned for Northants/Cumbria splits | 777 (target: 781) |
| Bug found: apportioned rows kept parent's `old_la_code`, causing cross-matched joins | 777 (unchanged — masked by a second issue) |
| Leftover suppressed duplicate rows dropped | **777 (confirmed correct)** |

No local authority year is silently missing or double-counted across the full 2013–2024 run. Every one of the remaining un-joined LA-years traces to a specific, documented cause: DfE's own footnoted data gaps (Hackney 2021/2022, Hampshire 2024) or genuine reorganisation timing differences between the two datasets, not an artefact of this pipeline.

`pathway_final` is now the clean base for the actual pathway analysis — referral → assessment → CIN → CPP → CLA — across local authorities and time.

## 4. Second bug found and fixed: stale `new_la_code`, and leftover pre-split placeholder rows

While building a year-on-year divergence table, `set_index("la_name_referrals")` failed with a duplicate-labels error. Investigating surfaced two related problems, both variants of the same root cause as the `old_la_code` bug: apportioned rows were built with `row.copy()`, which carries over every column from the row they were copied from unless explicitly overwritten.

1. **`new_la_code` was never fixed** alongside `old_la_code` — apportioned rows had the correct successor name and `old_la_code`, but still carried the parent county's ONS code (e.g. North Northamptonshire showing `E10000021`, Northamptonshire's code, instead of its own `E06000061`). Didn't break the join (joins used `old_la_code`), but was quietly wrong data sitting in the table.
2. **DfE's own dataset already had placeholder rows** for the successor authorities in the pre-split years — correctly suppressed (`x`/`z`), since those authorities didn't legally exist yet. These were never dropped before our apportioned rows were added, so each affected authority ended up with two rows for the same year: DfE's suppressed placeholder and our real apportioned figure.

Both fixed by: (a) adding an explicit `new_la_code` assignment to both apportion functions, and (b) dropping any pre-existing rows under the successor names in the years being apportioned, before concatenating the new rows in.

In [ ]:
SUCCESSOR_CODES = {
    "North Northamptonshire": 940,
    "West Northamptonshire": 941,
    "Cumberland": 942,
    "Westmorland and Furness": 943,
}

SUCCESSOR_NEW_CODES = {
    "North Northamptonshire": "E06000061",
    "West Northamptonshire": "E06000062",
    "Cumberland": "E06000063",
    "Westmorland and Furness": "E06000064",
}

def apportion_row(row, share_a, share_b, name_a, name_b):
    """Split a combined-authority row into two new rows using a fixed population share."""
    original_number = int(row["number"])

    row_a = row.copy()
    row_a["la_name"] = name_a
    row_a["old_la_code"] = SUCCESSOR_CODES[name_a]
    row_a["new_la_code"] = SUCCESSOR_NEW_CODES[name_a]
    row_a["number"] = round(original_number * share_a)

    row_b = row.copy()
    row_b["la_name"] = name_b
    row_b["old_la_code"] = SUCCESSOR_CODES[name_b]
    row_b["new_la_code"] = SUCCESSOR_NEW_CODES[name_b]
    row_b["number"] = round(original_number * share_b)

    return row_a, row_b


def apportion_years_col(df, county_name, years, share_a, share_b, name_a, name_b, col):
    combined = df[(df["la_name"] == county_name) & (df["time_period"].isin(years))]
    new_rows = []
    for _, row in combined.iterrows():
        original = int(row[col])
        row_a = row.copy()
        row_a["la_name"] = name_a
        row_a["old_la_code"] = SUCCESSOR_CODES[name_a]
        row_a["new_la_code"] = SUCCESSOR_NEW_CODES[name_a]
        row_a[col] = round(original * share_a)

        row_b = row.copy()
        row_b["la_name"] = name_b
        row_b["old_la_code"] = SUCCESSOR_CODES[name_b]
        row_b["new_la_code"] = SUCCESSOR_NEW_CODES[name_b]
        row_b[col] = round(original * share_b)

        new_rows.extend([row_a, row_b])
    return pd.DataFrame(new_rows)

### Full rebuild, in order

Each apportionment bug has surfaced through a chain of dependent cells, so the pipeline is re-run in full from the fixed functions downward rather than patched in place — partial re-runs are exactly how the earlier duplicate crept in unnoticed.

In [ ]:
# CLA apportionment
northants_apportioned = apportion_years(
    cla_snapshot, "Northamptonshire", [2020, 2021],
    north_share, west_share, "North Northamptonshire", "West Northamptonshire"
)
cumbria_apportioned = apportion_years(
    cla_snapshot, "Cumbria", [2020, 2021, 2022, 2023],
    cumberland_share, westmorland_share, "Cumberland", "Westmorland and Furness"
)

# Drop old combined rows AND any pre-existing suppressed successor placeholder rows
cla_snapshot_fixed = cla_snapshot[
    ~(
        ((cla_snapshot["la_name"] == "Northamptonshire") & (cla_snapshot["time_period"].isin([2020, 2021]))) |
        ((cla_snapshot["la_name"] == "Cumbria") & (cla_snapshot["time_period"].isin([2020, 2021, 2022, 2023])))
    )
]

years_to_clean = {
    "North Northamptonshire": [2020, 2021],
    "West Northamptonshire": [2020, 2021],
    "Cumberland": [2020, 2021, 2022, 2023],
    "Westmorland and Furness": [2020, 2021, 2022, 2023],
}
for name, years in years_to_clean.items():
    cla_snapshot_fixed = cla_snapshot_fixed[
        ~((cla_snapshot_fixed["la_name"] == name) & (cla_snapshot_fixed["time_period"].isin(years)))
    ]

cla_snapshot_fixed = pd.concat([cla_snapshot_fixed, northants_apportioned, cumbria_apportioned], ignore_index=True)

# Drop leftover suppressed old-name rows
leftover_suppressed = cla_snapshot_fixed[
    (cla_snapshot_fixed["la_name"].isin(["Northamptonshire", "Cumbria"])) &
    (cla_snapshot_fixed["number"] == "z")
]
cla_snapshot_fixed = cla_snapshot_fixed[~cla_snapshot_fixed.index.isin(leftover_suppressed.index)]

# Referrals apportionment
northants_ref_apportioned = apportion_years_col(
    la_rows, "Northamptonshire", list(range(2013, 2022)),
    north_share, west_share, "North Northamptonshire", "West Northamptonshire", "Referrals"
)
cumbria_ref_apportioned = apportion_years_col(
    la_rows, "Cumbria", list(range(2013, 2024)),
    cumberland_share, westmorland_share, "Cumberland", "Westmorland and Furness", "Referrals"
)

la_rows_fixed = la_rows[
    ~(
        ((la_rows["la_name"] == "Northamptonshire") & (la_rows["time_period"].isin(range(2013, 2022)))) |
        ((la_rows["la_name"] == "Cumbria") & (la_rows["time_period"].isin(range(2013, 2024))))
    )
]
la_rows_fixed = pd.concat([la_rows_fixed, northants_ref_apportioned, cumbria_ref_apportioned], ignore_index=True)

# Final join
pathway_final = la_rows_fixed.merge(
    cla_snapshot_fixed,
    on=["old_la_code", "time_period"],
    how="inner",
    suffixes=("_referrals", "_cla")
)

print(la_rows_fixed.shape[0], "referral rows")
print(cla_snapshot_fixed.shape[0], "CLA rows")
print(pathway_final.shape[0], "joined rows")

In [ ]:
# Confirm no duplicates remain
pathway_final[pathway_final["time_period"] == 2020]["la_name_referrals"].value_counts().head(6)

## 5. Scope correction: this is a 2020–2024 pathway, not 2013–2024

While building the divergence analysis it became clear `pathway_final` only ever contained five years (2020–2024), not the full referrals history back to 2013. Checking directly confirmed the cause: the CLA-by-local-authority table itself is only published by DfE from 2020 onward — a genuine limitation of the source data, not a join bug. (The Northamptonshire/Cumbria apportionment work above still correctly spans back to 2013 on the referrals side, and remains valid — it just can't extend the CLA side beyond what DfE has published.)

Every year-on-year comparison below should be read as a 2020–2024 picture, not a full decade.

## 6. Referral-to-CLA conversion rate, and finding a real outlier

`referral_to_cla_rate = number / Referrals * 100` — an approximate proxy, not a literal conversion percentage, since `Referrals` counts events across the year (a child can be referred more than once) while `number` is a point-in-time snapshot of children looked after on 31 March. Treated here as a comparative indicator across LAs and years, not a precise rate.

In [ ]:
pathway_final["Referrals"] = pd.to_numeric(pathway_final["Referrals"], errors="coerce")
pathway_final["number"] = pd.to_numeric(pathway_final["number"], errors="coerce")
pathway_final["referral_to_cla_rate"] = pathway_final["number"] / pathway_final["Referrals"] * 100

# Filter out small-denominator noise (e.g. City of London, population in the low thousands)
pathway_final[pathway_final["Referrals"] >= 500][
    ["time_period", "la_name_referrals", "Referrals", "number", "referral_to_cla_rate"]
].sort_values("referral_to_cla_rate", ascending=False).head(10)

**Shropshire** stands out: appears three times in the top 10 (2022, 2023, 2024), not a single-year fluke. Its five-year trend (2020→2024) shows referrals falling (1882→1522, -19%) while CLA numbers nearly double (399→717, +80%) — a consistent, monotonic move in both directions across all five years. Against a national average that barely moves (13.5% in 2020 to 14.6% in 2024), Shropshire goes from 21% to 47% over the same period — roughly triple the national rate by 2024, and the gap widening every year. A genuine local outlier, not part of a national trend Shropshire is simply further along.

Possible explanations worth investigating further: referral threshold tightening, an Ofsted inspection or serious case review triggering practice change, or financial/demand-management pressure on referral acceptance. Not yet determined which — next step is checking Shropshire's Ofsted history and whether other authorities (Liverpool appeared twice in the same top 10) show the same divergence pattern.

## 7. Systematic divergence check across all local authorities

Rather than treat Shropshire's pattern as a one-off spotted by eye, the same referrals-vs-CLA comparison is run across every local authority (2020 → 2024), to see whether Shropshire is a genuine outlier or the extreme end of a common pattern.

In [ ]:
def get_year_values(df, year, col):
    return df[df["time_period"] == year].set_index("la_name_referrals")[col]

first_referrals = get_year_values(pathway_final, 2020, "Referrals")
last_referrals = get_year_values(pathway_final, 2024, "Referrals")
first_cla = get_year_values(pathway_final, 2020, "number")
last_cla = get_year_values(pathway_final, 2024, "number")

divergence = pd.DataFrame({
    "referrals_2020": first_referrals,
    "referrals_2024": last_referrals,
    "cla_2020": first_cla,
    "cla_2024": last_cla,
}).dropna()

divergence["referrals_pct_change"] = (divergence["referrals_2024"] - divergence["referrals_2020"]) / divergence["referrals_2020"] * 100
divergence["cla_pct_change"] = (divergence["cla_2024"] - divergence["cla_2020"]) / divergence["cla_2020"] * 100

print(len(divergence), "local authorities in the comparison")
divergence.sort_values("cla_pct_change", ascending=False).head(10)

**Finding: "referrals falling while CLA rises" is a common pattern (41 of 150 LAs show it), so that combination alone isn't distinctive.** What sets Shropshire apart is the *scale* of its CLA growth specifically — 79.7%, far ahead of the next-highest authority (Wokingham, 39.8%) — regardless of what referrals are doing elsewhere. Several other high-CLA-growth authorities (Halton, Barnsley, Stockport, Bexley, Cornwall) show *rising* referrals, so rapid CLA growth isn't tied to falling referrals nationally. Shropshire's specific combination — CLA nearly doubling while referrals actually declined — is rarer than either trend alone.

In [ ]:
print(len(divergence), "total local authorities")
divergence["cla_pct_change"].describe()

### Confirming Shropshire as a statistical outlier, not just top-of-table

Across 150 local authorities, CLA growth 2020→2024 has a mean of 3.9%, std dev 16.9%, and an interquartile range of 18.9 points (25th percentile -5.8%, 75th percentile 13.1%).

- **Standard IQR outlier fence** (75th percentile + 1.5×IQR): ~41.4%. Shropshire's 79.7% clears this by close to double.
- **In standard deviations**: Shropshire sits roughly **4.5 standard deviations above the mean** — a genuine statistical outlier by any conventional threshold, not an impression from eyeballing a sorted table.

## Finding, dated and summarised

**Shropshire's Children Looked After numbers grew 79.7% between 2020 and 2024 (399 → 717) while referrals to children's social care fell 19.1% (1,882 → 1,522) over the same period — a divergence roughly 4.5 standard deviations beyond the distribution of England's other 149 local authorities.** The pattern (referrals down, CLA up) is not unusual in itself — 41 of 150 LAs show it — but the scale of Shropshire's CLA growth is unmatched by any other authority in the comparison.

**Not yet established:** the cause. Plausible explanations include referral-threshold tightening, a practice change following an Ofsted inspection or serious case review, or financial/demand-management pressure on referral acceptance — but none of these have been checked against the data yet. That's the next piece of work: Shropshire's Ofsted inspection history over 2020–2024, and whether the divergence coincides with any documented practice or leadership change.

**Scope reminder:** this finding is built on 2020–2024 data only (see Section 5) — the CLA-by-local-authority table isn't published by DfE further back, so this can't currently be extended into a longer historical trend.

## 8. Building the full funnel: adding CIN and assessment stages

With referrals and CLA joined (`pathway_final`), the middle of the pathway — Children in Need (CIN) and completed assessments — is added the same way, one dataset at a time, checking geographic level, category, year range, and the Northamptonshire/Cumbria gap for each before joining.

Two more DfE tables:
- **B1** (Children in need and episodes of need by local authority) — `At31_episodes` gives the CIN stock count at 31 March, same point-in-time logic as the CLA snapshot.
- **C2** (Completed assessments by duration and local authority) — `Assessments_completed` gives the assessment stage count.

Both show the same Northamptonshire/Cumbria combined-then-split pattern as referrals and CLA, fixed the same way each time with the existing `apportion_years_col` function and the same two population ratios calculated earlier — no new logic needed, just reapplying an established pattern to two more datasets.

In [ ]:
# B1: Children in need
b1_url = "https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/37cb9a2a-7e3f-409e-8dc4-1fd399ee7d32/csv"
df_b1 = pd.read_csv(b1_url)
b1_la = df_b1[df_b1["geographic_level"] == "Local authority"]

northants_cin_apportioned = apportion_years_col(
    b1_la, "Northamptonshire", [2020, 2021],
    north_share, west_share, "North Northamptonshire", "West Northamptonshire", "At31_episodes"
)
cumbria_cin_apportioned = apportion_years_col(
    b1_la, "Cumbria", [2020, 2021, 2022, 2023],
    cumberland_share, westmorland_share, "Cumberland", "Westmorland and Furness", "At31_episodes"
)

b1_la_fixed = b1_la[
    ~(
        ((b1_la["la_name"] == "Northamptonshire") & (b1_la["time_period"].isin([2020, 2021]))) |
        ((b1_la["la_name"] == "Cumbria") & (b1_la["time_period"].isin([2020, 2021, 2022, 2023])))
    )
]
b1_la_fixed = pd.concat([b1_la_fixed, northants_cin_apportioned, cumbria_cin_apportioned], ignore_index=True)

pathway_with_cin = pathway_final.merge(
    b1_la_fixed[["old_la_code", "time_period", "At31_episodes"]],
    on=["old_la_code", "time_period"],
    how="left"
)
print(pathway_with_cin.shape[0], "rows,", pathway_with_cin["At31_episodes"].isna().sum(), "unmatched")

In [ ]:
# C2: Completed assessments
c2_url = "https://explore-education-statistics.service.gov.uk/data-catalogue/data-set/671a058e-8b8a-4cba-a8be-c9d3399c666e/csv"
df_c2 = pd.read_csv(c2_url)
c2_la = df_c2[df_c2["geographic_level"] == "Local authority"]

northants_assess_apportioned = apportion_years_col(
    c2_la, "Northamptonshire", [2020, 2021],
    north_share, west_share, "North Northamptonshire", "West Northamptonshire", "Assessments_completed"
)
cumbria_assess_apportioned = apportion_years_col(
    c2_la, "Cumbria", [2020, 2021, 2022, 2023],
    cumberland_share, westmorland_share, "Cumberland", "Westmorland and Furness", "Assessments_completed"
)

c2_la_fixed = c2_la[
    ~(
        ((c2_la["la_name"] == "Northamptonshire") & (c2_la["time_period"].isin([2020, 2021]))) |
        ((c2_la["la_name"] == "Cumbria") & (c2_la["time_period"].isin([2020, 2021, 2022, 2023])))
    )
]
c2_la_fixed = pd.concat([c2_la_fixed, northants_assess_apportioned, cumbria_assess_apportioned], ignore_index=True)

pathway_with_assess = pathway_with_cin.merge(
    c2_la_fixed[["old_la_code", "time_period", "Assessments_completed"]],
    on=["old_la_code", "time_period"],
    how="left"
)

print(pathway_with_assess.shape[0], "rows")
pathway_with_assess[pathway_with_assess["Assessments_completed"].isna()]["time_period"].value_counts()

**Result: every unmatched row is in 2024 (153 rows), nothing else.** C2 (assessments) is only published by DfE up to 2023 — one year behind referrals, CIN, and CLA, which all extend to 2024. This is a genuine data-availability limit, not a join problem: confirmed by checking the unmatched-row breakdown showed exactly the pattern predicted (all in 2024, none scattered elsewhere) before accepting it.

**Scope note:** the assessment stage of the funnel covers 2020–2023 only. Any view of the full five-stage funnel (referral → assessment → CIN → CPP → CLA) needs to either exclude 2024 or show assessments as missing for that year — stated explicitly here rather than silently handled by a join that happens to produce NaN.